<a href="https://colab.research.google.com/github/Ricardo-web-ctrl/CryptoBot_AI_v1.ipynb/blob/main/CryptoBot_AI_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# CRYPTOBOT AI - INSTALACIÓN DE HERRAMIENTAS
# Ejecuta esta celda una sola vez al iniciar el entorno.

!pip -q install pandas numpy requests matplotlib

print("================================")
print("     CRYPTOBOT AI v1.0")
print("================================")
print("Herramientas instaladas correctamente.")
print("Entorno: Google Colab")
print("Modo: DEMO / SIMULACIÓN")


     CRYPTOBOT AI v1.0
Herramientas instaladas correctamente.
Entorno: Google Colab
Modo: DEMO / SIMULACIÓN


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

# ==========================================
# CRYPTOBOT AI - MÓDULO 1
# CONEXIÓN CON BINANCE Y ANÁLISIS DE BTC
# ==========================================

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timezone

# CONFIGURACIÓN
BASE_URL = "https://data-api.binance.vision"
SYMBOL = "BTCUSDT"
TIMEFRAME = "15m"

print("Iniciando CryptoBot AI...")

# 1. OBTENER VELAS DE BINANCE
url = f"{BASE_URL}/api/v3/klines"

params = {
    "symbol": SYMBOL,
    "interval": TIMEFRAME,
    "limit": 200
}

response = requests.get(url, params=params, timeout=20)
response.raise_for_status()

data = response.json()

# 2. ORGANIZAR LOS DATOS
columns = [
    "time", "open", "high", "low", "close",
    "volume", "close_time", "quote_volume",
    "trades", "taker_buy", "taker_quote", "ignore"
]

df = pd.DataFrame(data, columns=columns)

# Convertir precios y volumen a números
for col in ["open", "high", "low", "close", "volume"]:
    df[col] = pd.to_numeric(df[col])

df["time"] = pd.to_datetime(df["time"], unit="ms", utc=True)

# Excluir la vela que todavía no ha cerrado
df = df.iloc[:-1].copy()

# 3. CALCULAR MEDIAS MÓVILES
df["EMA21"] = df["close"].ewm(
    span=21, adjust=False
).mean()

df["EMA50"] = df["close"].ewm(
    span=50, adjust=False
).mean()

# 4. CALCULAR RSI 14
delta = df["close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.ewm(
    alpha=1/14, adjust=False, min_periods=14
).mean()

avg_loss = loss.ewm(
    alpha=1/14, adjust=False, min_periods=14
).mean()

rs = avg_gain / avg_loss.replace(0, np.nan)
df["RSI14"] = 100 - (100 / (1 + rs))

# 5. ANALIZAR LA ÚLTIMA VELA CERRADA
last = df.iloc[-1]

price = last["close"]
ema21 = last["EMA21"]
ema50 = last["EMA50"]
rsi = last["RSI14"]

# 6. GENERAR SEÑAL INICIAL
if ema21 > ema50 and 50 < rsi < 70:
    signal = "POSIBLE LONG"

elif ema21 < ema50 and 30 < rsi < 50:
    signal = "POSIBLE SHORT"

else:
    signal = "ESPERAR"

# 7. MOSTRAR RESULTADOS
print("\n========== RESULTADOS ==========")

print("Mercado:", SYMBOL)
print("Temporalidad:", TIMEFRAME)
print("Última vela cerrada:", last["time"])
print("Precio BTC:", round(price, 2))
print("EMA 21:", round(ema21, 2))
print("EMA 50:", round(ema50, 2))
print("RSI 14:", round(rsi, 2))

print("\nSEÑAL DEL ALGORITMO:", signal)

print("\nModo: SIMULACIÓN")
print("Órdenes reales: DESACTIVADAS")
print("Saldo utilizado: 0 USDT")

print("\nCryptoBot AI ejecutado correctamente.")


Iniciando CryptoBot AI...

========== RESULTADOS ==========
Mercado: BTCUSDT
Temporalidad: 15m
Última vela cerrada: 2026-11-25 01:45:00+00:00
Precio BTC: 59272.01
EMA 21: 59686.3
EMA 50: 59860.88
RSI 14: 33.09

SEÑAL DEL ALGORITMO: POSIBLE SHORT

Modo: SIMULACIÓN
Órdenes reales: DESACTIVADAS
Saldo utilizado: 0 USDT

CryptoBot AI ejecutado correctamente.
